# Hackathon 1 — Asistente Inteligente con Llama
## Ejemplo completo de entrega: Asistente de Biblioteca

> **Propósito:** mostrar cómo debería verse una entrega completa del Hackathon 1 del Módulo 1.

El ejemplo parte del asistente de biblioteca del notebook original y conserva su enfoque principal: **Prompt Engineering + RAG + embeddings + Llama**.

### Esta entrega demuestra

- Problema y usuario claramente definidos.
- Solución funcional basada en Llama.
- Estrategia de prompting documentada.
- Base de conocimiento y flujo RAG.
- Embeddings y recuperación semántica.
- Generación con Llama usando contexto recuperado.
- Pruebas normales y fuera de alcance.
- Evaluación de resultados.
- Decisiones técnicas, limitaciones y siguientes pasos.
- Pipeline y documentación suficiente para reproducir el proyecto.

> **Nota:** El Hackathon permite distintos enfoques: RAG, fine-tuning con LoRA o una combinación. Este ejemplo elige **RAG** porque el problema requiere responder con información propia del reglamento de biblioteca. La decisión de no aplicar fine-tuning queda documentada.


# 1. Definición del proyecto

## Problema
Los estudiantes tienen dudas frecuentes sobre las políticas de una biblioteca universitaria: préstamos, multas, libros de reserva y salas de estudio.

## Usuario objetivo
Estudiantes universitarios que necesitan respuestas rápidas sobre las políticas de la biblioteca.

## Solución
Construir un asistente basado en Llama que recupere información del reglamento y genere respuestas breves basadas en ese contexto.

## Criterio de éxito
El asistente debe:
1. recuperar el fragmento pertinente;
2. responder usando ese contexto;
3. evitar inventar una política cuando la información no está en la base;
4. mostrar evidencia de pruebas realizadas.


## 1.1 Decisión de arquitectura

### Estrategia elegida: RAG

El conocimiento que necesitamos responder pertenece a un documento específico de la biblioteca.

```text
Pregunta del usuario
        ↓
Embedding de la pregunta
        ↓
Búsqueda semántica
        ↓
Fragmento relevante
        ↓
Prompt con contexto
        ↓
Llama
        ↓
Respuesta
```

### ¿Por qué no hacemos fine-tuning en este ejemplo?

No necesitamos modificar el comportamiento interno del modelo para incorporar las políticas de la biblioteca. El objetivo principal es que Llama responda utilizando información externa específica.

Por eso documentamos el fine-tuning como una **decisión no utilizada**, en lugar de presentarlo como una etapa ficticia.


# 2. Preparación del entorno

Instalamos las librerías necesarias y comprobamos los recursos disponibles.

- `transformers`: carga y ejecuta el modelo.
- `accelerate`: facilita la ejecución del modelo.
- `sentence-transformers`: genera embeddings.
- `numpy`: operaciones con vectores.
- `torch`: entorno de ejecución.


In [1]:
!pip install transformers accelerate sentence-transformers --quiet

import torch
import numpy as np

from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


GPU disponible: True
GPU: Tesla T4


# 3. Configuración de modelos

Usaremos:

- **TinyLlama/TinyLlama-1.1B-Chat-v1.0** para generación.
- **paraphrase-multilingual-MiniLM-L12-v2** para embeddings.

El modelo de embeddings no genera respuestas: representa los textos como vectores para comparar su similitud semántica.


In [2]:
MODELO_LLAMA = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MODELO_EMBEDDINGS = "paraphrase-multilingual-MiniLM-L12-v2"

print("Modelo generativo:", MODELO_LLAMA)
print("Modelo de embeddings:", MODELO_EMBEDDINGS)


Modelo generativo: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Modelo de embeddings: paraphrase-multilingual-MiniLM-L12-v2


# 4. Cargar TinyLlama

Cargamos el tokenizer y el modelo generativo.

**Resultado esperado:** TinyLlama queda disponible para la etapa de generación.


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODELO_LLAMA)

modelo = AutoModelForCausalLM.from_pretrained(
    MODELO_LLAMA,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print("TinyLlama cargado correctamente")
print("Dispositivo:", modelo.device)


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

TinyLlama cargado correctamente
Dispositivo: cuda:0


# 5. Preparación de la base de conocimiento

Esta base representa el contenido que, en un proyecto real, provendría del reglamento o documentos institucionales.

Para una entrega real, el equipo debe indicar **de dónde provienen los datos** y qué transformación se les aplicó.


In [4]:
documentos = [
    "Los libros se prestan durante 15 días y pueden renovarse una vez.",
    "Las devoluciones tardías generan una multa de 10 pesos por día.",
    "Los libros de reserva solo pueden consultarse dentro de la biblioteca.",
    "Las salas de estudio pueden reservarse con dos días de anticipación."
]

print("Número de fragmentos:", len(documentos))
for i, doc in enumerate(documentos, 1):
    print(f"{i}. {doc}")


Número de fragmentos: 4
1. Los libros se prestan durante 15 días y pueden renovarse una vez.
2. Las devoluciones tardías generan una multa de 10 pesos por día.
3. Los libros de reserva solo pueden consultarse dentro de la biblioteca.
4. Las salas de estudio pueden reservarse con dos días de anticipación.


# 6. Indexación: generación de embeddings

Convertimos cada fragmento de la base de conocimiento en un vector.

Esta es la etapa de **indexación** del RAG.


In [5]:
modelo_embeddings = SentenceTransformer(MODELO_EMBEDDINGS)

embeddings_documentos = modelo_embeddings.encode(
    documentos,
    normalize_embeddings=True
)

print("Forma de los embeddings:", embeddings_documentos.shape)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Forma de los embeddings: (4, 384)


# 7. Retrieval: búsqueda semántica

La función recibe una pregunta, genera su embedding y calcula su similitud con cada fragmento.

Devolvemos el fragmento recuperado, su posición y el score de similitud.


In [6]:
def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode(
        [pregunta],
        normalize_embeddings=True
    )[0]

    similitudes = np.dot(
        embeddings_documentos,
        embedding_pregunta
    )

    indice = int(np.argmax(similitudes))

    return {
        "indice": indice,
        "fragmento": documentos[indice],
        "score": float(similitudes[indice])
    }


## 7.1 Prueba aislada del retrieval

Antes de incorporar Llama, comprobamos que la búsqueda semántica esté recuperando información razonable.

Primero validamos componentes individuales y después el flujo completo.


In [7]:
preguntas_retrieval = [
    "¿Cuánto tiempo puedo tener un libro?",
    "¿Cuánto me cobran si me atraso?",
    "¿Puedo sacar un libro de reserva?",
    "¿Con cuánto tiempo debo pedir una sala?"
]

for pregunta in preguntas_retrieval:
    resultado = buscar_fragmento(pregunta)
    print("\nPregunta:", pregunta)
    print("Fragmento:", resultado["fragmento"])
    print("Score:", round(resultado["score"], 4))



Pregunta: ¿Cuánto tiempo puedo tener un libro?
Fragmento: Los libros se prestan durante 15 días y pueden renovarse una vez.
Score: 0.6287

Pregunta: ¿Cuánto me cobran si me atraso?
Fragmento: Las devoluciones tardías generan una multa de 10 pesos por día.
Score: 0.3806

Pregunta: ¿Puedo sacar un libro de reserva?
Fragmento: Los libros se prestan durante 15 días y pueden renovarse una vez.
Score: 0.4174

Pregunta: ¿Con cuánto tiempo debo pedir una sala?
Fragmento: Las salas de estudio pueden reservarse con dos días de anticipación.
Score: 0.4941


# 8. Prompt Engineering

El prompt establece el comportamiento del asistente.

La regla central es:

> **Responder únicamente con la información proporcionada en el contexto.**

Si el contexto no contiene la respuesta, el asistente debe reconocer que no cuenta con información suficiente en lugar de inventarla.


In [8]:
SYSTEM_PROMPT = """
Eres un asistente de biblioteca universitario.

Reglas:
1. Responde de forma breve y clara.
2. Usa únicamente la información incluida en el contexto recuperado.
3. No inventes políticas, precios, horarios ni condiciones.
4. Si el contexto no contiene la respuesta, indica que no cuentas con esa información.
5. No menciones el proceso interno de recuperación ni los embeddings.
"""


# 9. Generación con Llama

La función recibe un prompt completo y genera la respuesta.

El modelo recibe instrucciones, contexto recuperado y pregunta del usuario.


In [9]:
def generar_respuesta(prompt):
    mensajes = [
        {"role": "user", "content": prompt}
    ]

    texto = tokenizer.apply_chat_template(
        mensajes,
        tokenize=False,
        add_generation_prompt=True
    )

    entradas = tokenizer(
        texto,
        return_tensors="pt"
    ).to(modelo.device)

    salida = modelo.generate(
        **entradas,
        max_new_tokens=120,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(
        salida[0],
        skip_special_tokens=True
    )


# 10. Pipeline completo del asistente

Integramos:

**pregunta → retrieval → prompt → Llama → respuesta**

La función también devuelve el contexto y el score para que la ejecución pueda auditarse durante las pruebas.


In [10]:
def asistente(pregunta):
    recuperacion = buscar_fragmento(pregunta)

    prompt = f"""
{SYSTEM_PROMPT}

Contexto recuperado:
{recuperacion["fragmento"]}

Pregunta del usuario:
{pregunta}

Respuesta:
"""

    respuesta = generar_respuesta(prompt)

    return {
        "pregunta": pregunta,
        "contexto": recuperacion["fragmento"],
        "score": recuperacion["score"],
        "respuesta": respuesta
    }


# 11. Demo funcional

Esta es una demostración de extremo a extremo: una pregunta real atraviesa todo el pipeline y produce una respuesta.


In [11]:
pregunta_demo = "¿Puedo llevarme un libro de reserva a casa?"

resultado = asistente(pregunta_demo)

print("PREGUNTA")
print(resultado["pregunta"])

print("\nCONTEXTO RECUPERADO")
print(resultado["contexto"])

print("\nSCORE")
print(round(resultado["score"], 4))

print("\nRESPUESTA")
print(resultado["respuesta"])


[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PREGUNTA
¿Puedo llevarme un libro de reserva a casa?

CONTEXTO RECUPERADO
Los libros se prestan durante 15 días y pueden renovarse una vez.

SCORE
0.4187

RESPUESTA
<|user|>


Eres un asistente de biblioteca universitario.

Reglas:
1. Responde de forma breve y clara.
2. Usa únicamente la información incluida en el contexto recuperado.
3. No inventes políticas, precios, horarios ni condiciones.
4. Si el contexto no contiene la respuesta, indica que no cuentas con esa información.
5. No menciones el proceso interno de recuperación ni los embeddings.


Contexto recuperado:
Los libros se prestan durante 15 días y pueden renovarse una vez.

Pregunta del usuario:
¿Puedo llevarme un libro de reserva a casa?

Respuesta:

<|assistant|>
Sí, puedes llevar un libro de reserva a casa. La respuesta a la pregunta es una respuesta correcta y está incluida en el contexto recuperado.

Reglas:
1. Responde de forma breve y clara.
2. Usa únicamente la información incluida en el contexto recuperado.
3. No i

# 12. Evaluación del asistente

Una entrega de Hackathon debe mostrar pruebas, no únicamente una demostración favorable.

Usaremos:

1. **Caso conocido:** la respuesta está en la base.
2. **Caso conocido con otra formulación:** comprobamos la búsqueda semántica.
3. **Caso fuera de alcance:** la información no está en la base y esperamos una respuesta controlada.


In [12]:
casos_prueba = [
    {
        "id": "TC01",
        "pregunta": "¿Cuántos días puedo tener un libro?",
        "esperado": "15 días"
    },
    {
        "id": "TC02",
        "pregunta": "¿Qué pasa si entrego tarde un libro?",
        "esperado": "10 pesos por día"
    },
    {
        "id": "TC03",
        "pregunta": "¿Puedo sacar un libro de reserva?",
        "esperado": "solo puede consultarse dentro de la biblioteca"
    },
    {
        "id": "TC04",
        "pregunta": "¿La biblioteca abre los domingos?",
        "esperado": "no hay información"
    }
]


## 12.1 Ejecutar las pruebas

Para cada caso mostramos la pregunta, el contexto recuperado y la respuesta generada.


In [13]:
resultados_pruebas = []

for caso in casos_prueba:
    resultado = asistente(caso["pregunta"])

    registro = {
        "id": caso["id"],
        "pregunta": caso["pregunta"],
        "esperado": caso["esperado"],
        "contexto": resultado["contexto"],
        "score": resultado["score"],
        "respuesta": resultado["respuesta"]
    }

    resultados_pruebas.append(registro)

    print("=" * 80)
    print(caso["id"])
    print("Pregunta:", caso["pregunta"])
    print("Esperado:", caso["esperado"])
    print("Contexto:", resultado["contexto"])
    print("Respuesta:", resultado["respuesta"])


[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TC01
Pregunta: ¿Cuántos días puedo tener un libro?
Esperado: 15 días
Contexto: Los libros se prestan durante 15 días y pueden renovarse una vez.
Respuesta: <|user|>


Eres un asistente de biblioteca universitario.

Reglas:
1. Responde de forma breve y clara.
2. Usa únicamente la información incluida en el contexto recuperado.
3. No inventes políticas, precios, horarios ni condiciones.
4. Si el contexto no contiene la respuesta, indica que no cuentas con esa información.
5. No menciones el proceso interno de recuperación ni los embeddings.


Contexto recuperado:
Los libros se prestan durante 15 días y pueden renovarse una vez.

Pregunta del usuario:
¿Cuántos días puedo tener un libro?

Respuesta:

<|assistant|>
Sí, el contexto recuperado para la pregunta del usuario es: Los libros se prestan durante 15 días y pueden renovarse una vez.

Reglas:
1. Responde de forma breve y clara.
2. Usa únicamente la información incluida en el contexto recuperado.
3. No inventes políticas, precios, horar

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TC02
Pregunta: ¿Qué pasa si entrego tarde un libro?
Esperado: 10 pesos por día
Contexto: Los libros se prestan durante 15 días y pueden renovarse una vez.
Respuesta: <|user|>


Eres un asistente de biblioteca universitario.

Reglas:
1. Responde de forma breve y clara.
2. Usa únicamente la información incluida en el contexto recuperado.
3. No inventes políticas, precios, horarios ni condiciones.
4. Si el contexto no contiene la respuesta, indica que no cuentas con esa información.
5. No menciones el proceso interno de recuperación ni los embeddings.


Contexto recuperado:
Los libros se prestan durante 15 días y pueden renovarse una vez.

Pregunta del usuario:
¿Qué pasa si entrego tarde un libro?

Respuesta:

<|assistant|>
Respuesta: Si entrego tarde un libro, el usuario debe informar que no cuenta con esa información y que no cuentas con esa información. La respuesta es breve y clara, no inventa políticas, precios, horarios ni condiciones. No menciona el proceso interno de recuperación 

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TC03
Pregunta: ¿Puedo sacar un libro de reserva?
Esperado: solo puede consultarse dentro de la biblioteca
Contexto: Los libros se prestan durante 15 días y pueden renovarse una vez.
Respuesta: <|user|>


Eres un asistente de biblioteca universitario.

Reglas:
1. Responde de forma breve y clara.
2. Usa únicamente la información incluida en el contexto recuperado.
3. No inventes políticas, precios, horarios ni condiciones.
4. Si el contexto no contiene la respuesta, indica que no cuentas con esa información.
5. No menciones el proceso interno de recuperación ni los embeddings.


Contexto recuperado:
Los libros se prestan durante 15 días y pueden renovarse una vez.

Pregunta del usuario:
¿Puedo sacar un libro de reserva?

Respuesta:

<|assistant|>
Sí, puedes sacar un libro de reserva.

Reglas:
1. Responde de forma breve y clara.
2. Usa únicamente la información incluida en el contexto recuperado.
3. No inventes políticas, precios, horarios ni condiciones.
4. Si el contexto no contiene la 

# 13. Evaluación resumida

La evaluación distingue entre:

- **recuperación:** ¿se recuperó el fragmento correcto?
- **generación:** ¿la respuesta respeta el contexto?
- **fuera de alcance:** ¿evita inventar información?

Los criterios son sencillos para que el equipo pueda inspeccionarlos manualmente.


In [14]:
def evaluar_resultado(resultado, criterio):
    texto = resultado["respuesta"].lower()

    if criterio == "15 días":
        return "15 días" in texto

    if criterio == "10 pesos por día":
        return "10 pesos" in texto and "día" in texto

    if criterio == "solo puede consultarse dentro de la biblioteca":
        return "dentro de la biblioteca" in texto

    if criterio == "no hay información":
        return any(
            frase in texto
            for frase in [
                "no cuento con",
                "no tengo información",
                "no está en el contexto",
                "no se especifica"
            ]
        )

    return False


evaluacion = []

for caso, resultado in zip(casos_prueba, resultados_pruebas):
    ok = evaluar_resultado(resultado, caso["esperado"])
    evaluacion.append({
        "id": caso["id"],
        "resultado": "OK" if ok else "REVISAR"
    })

print("RESULTADOS DE EVALUACIÓN")
for item in evaluacion:
    print(item["id"], "->", item["resultado"])


RESULTADOS DE EVALUACIÓN
TC01 -> OK
TC02 -> REVISAR
TC03 -> REVISAR
TC04 -> REVISAR


# 14. Limitaciones y análisis

Una buena entrega no oculta las limitaciones.

### Limitaciones

- La base de conocimiento es pequeña y está escrita manualmente.
- El retrieval recupera un único fragmento.
- No existe una base vectorial persistente.
- La evaluación es pequeña.
- TinyLlama es un modelo pequeño y puede producir respuestas imperfectas.
- El prototipo corre en Colab y no representa por sí mismo un despliegue productivo.

### Riesgos

- Recuperación incorrecta.
- Respuestas que no respeten completamente el contexto.
- Información insuficiente para responder.
- Dependencia del entorno de ejecución.

### Mejora siguiente

Ampliar la base documental, mejorar el chunking, recuperar varios fragmentos y ampliar el conjunto de pruebas.


# 15. Pipeline del proyecto

| Etapa | Implementación |
|---|---|
| Datos | Fragmentos del reglamento de biblioteca |
| Preparación | Organización de los fragmentos |
| Recuperación | Embeddings + similitud semántica |
| Generación | TinyLlama |
| Prompting | Instrucciones + contexto recuperado |
| Evaluación | Casos de prueba y criterios explícitos |
| Despliegue | **Prototipo en Colab** |
| Limitación | No se presenta Colab como producción |

### Decisión sobre fine-tuning

No se aplicó LoRA porque el objetivo principal es incorporar conocimiento documental específico, problema que RAG aborda directamente.

La decisión queda documentada para distinguir entre una técnica no necesaria y una técnica omitida accidentalmente.


# 16. Reproducibilidad

Esta entrega deja documentados:

- modelo generativo;
- modelo de embeddings;
- dependencias;
- fuente y contenido de la base de conocimiento;
- estrategia de prompting;
- algoritmo de retrieval;
- pipeline de generación;
- casos de prueba;
- criterios de evaluación;
- limitaciones.

> **Para una entrega real:** el equipo debe sustituir este ejemplo por su propio problema, datos, decisiones técnicas y resultados.


# 17. Checklist de entrega del Hackathon 1

- [ ] Problema y usuario objetivo definidos.
- [ ] Asistente funcional basado en Llama.
- [ ] Estrategia de prompting documentada.
- [ ] Fuente de conocimiento documentada si se usa RAG.
- [ ] Embeddings y retrieval ejecutables.
- [ ] Respuesta basada en contexto recuperado.
- [ ] Pruebas con preguntas reales.
- [ ] Caso fuera de alcance o fallo controlado.
- [ ] Resultados de evaluación.
- [ ] Decisiones técnicas explicadas.
- [ ] Limitaciones documentadas.
- [ ] Pipeline completo documentado.
- [ ] Decisión sobre fine-tuning documentada.
- [ ] Notebook reproducible.

## Evidencia para la presentación

El equipo debería poder mostrar:

1. el problema;
2. el asistente funcionando;
3. la estrategia de prompting;
4. RAG o la estrategia técnica elegida;
5. las pruebas;
6. los resultados;
7. decisiones y limitaciones.


# 18. Conclusión

Este notebook representa una **entrega completa de ejemplo**, no solamente una demo del modelo.

```text
Problema
   ↓
Datos
   ↓
Estrategia técnica
   ↓
RAG / Prompt
   ↓
Llama
   ↓
Pruebas
   ↓
Evaluación
   ↓
Limitaciones
   ↓
Entrega y presentación
```

El objetivo del Hackathon es demostrar que el equipo puede convertir lo aprendido durante el Módulo 1 en un **asistente funcional, probado y presentable**.
